In [2]:
import os
import json
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

import optuna
import optuna.logging
optuna.logging.set_verbosity(optuna.logging.CRITICAL)

from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (
    RandomForestRegressor, 
    GradientBoostingRegressor,
    ExtraTreesRegressor,
    HistGradientBoostingRegressor,
    StackingRegressor
)
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.feature_selection import SelectFromModel, RFE
from sklearn.metrics import (
    mean_absolute_error,
    r2_score,
    mean_absolute_percentage_error,
    mean_squared_error
)

from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor

import shap

from utils.utils import (
    connection,
    data_from_ticker,
    data_from_tpulse,
    data_from_macrofactors,
)
import utils.final_config_ml as final_config_ml

In [3]:
companies = pd.read_sql("SELECT * FROM companies", connection())
tickers = companies['ticker'].tolist()
left_date = '2026-01-12'
right_date = '2026-03-12'
train_period = final_config_ml.TRAIN_PERIOD
val_period = final_config_ml.VAL_PERIOD
test_period = final_config_ml.TEST_PERIOD
step = final_config_ml.STEP
n_trials = final_config_ml.N_TRIALS
metric_optuna = final_config_ml.METRIC_OPTUNA
top_n_features = final_config_ml.TOP_N_FEATURES
MODELS_CONFIG = final_config_ml.MODELS_CONFIG

In [4]:
def create_technical_indicators(df):
    """Создание технических индикаторов для улучшения предсказаний"""
    df = df.copy()
    
    # Скользящие средние
    for window in [5, 10, 20]:
        if 'open' in df.columns:
            df[f'open_ma_{window}'] = df['open'].rolling(window=window).mean()
            df[f'open_ma_{window}_diff'] = df['open'] - df[f'open_ma_{window}']
    
    # Скользящие стандартные отклонения (волатильность)
    for window in [5, 10, 20]:
        if 'open' in df.columns:
            df[f'open_std_{window}'] = df['open'].rolling(window=window).std()
    
    # RSI (Relative Strength Index) - упрощенная версия
    if 'close' in df.columns and 'open' in df.columns:
        delta = df['close'].diff()
        gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
        loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
        rs = gain / (loss + 1e-10)
        df['rsi_14'] = 100 - (100 / (1 + rs))
    
    # MACD-like индикатор
    if 'close' in df.columns:
        exp1 = df['close'].ewm(span=12, adjust=False).mean()
        exp2 = df['close'].ewm(span=26, adjust=False).mean()
        df['macd'] = exp1 - exp2
        df['macd_signal'] = df['macd'].ewm(span=9, adjust=False).mean()
        df['macd_hist'] = df['macd'] - df['macd_signal']
    
    # Лаги для target
    if 'target' in df.columns:
        for lag in [1, 2, 3, 5, 7]:
            df[f'target_lag_{lag}'] = df['target'].shift(lag)
    
    # Процентные изменения
    if 'target' in df.columns:
        for period in [1, 3, 5, 10]:
            df[f'target_pct_change_{period}'] = df['target'].pct_change(periods=period)
    
    # Мин/Макс за период
    for window in [5, 10, 20]:
        if 'open' in df.columns:
            df[f'open_min_{window}'] = df['open'].rolling(window=window).min()
            df[f'open_max_{window}'] = df['open'].rolling(window=window).max()
            df[f'open_range_{window}'] = df[f'open_max_{window}'] - df[f'open_min_{window}']
    
    return df

In [5]:
def pack_all_data_for_ml_models(ticker: str, left_date: str, right_date: str, conn):
    """Сбор данных с правильной очисткой"""
    tpulse_data = data_from_tpulse(ticker, left_date, right_date, conn)
    ticker_data = data_from_ticker(ticker, left_date, right_date, conn)
    macrofactor_data = data_from_macrofactors(ticker, left_date, right_date, conn)

    data = tpulse_data.merge(macrofactor_data, how='left', on=['dt', 'ticker']).merge(
        ticker_data, how='left', on=['dt', 'ticker']
    )
    data = data[~data['target'].isnull()]
    
    # Добавление технических индикаторов
    data = create_technical_indicators(data)
    
    # Заполнение пропусков
    numeric_cols = data.select_dtypes(include=[np.number]).columns
    data[numeric_cols] = data[numeric_cols].fillna(method='ffill').fillna(method='bfill').fillna(0)
    
    return data

In [6]:
def sliding_windows_cross_validatin(df, train_days, val_days, test_days, step):
    """Кросс-валидация скользящим окном с проверкой на достаточность данных"""
    windows = []
    n = len(df)
    current_test_end = n
    
    min_required = train_days + val_days + test_days
    
    while True:
        test_start = current_test_end - test_days
        if test_start < 0:
            break
        val_start = test_start - val_days
        if val_start < 0:
            break
        train_start = val_start - train_days
        if train_start < 0:
            break
        
        # Проверка на достаточность данных
        if len(df.iloc[train_start:val_start]) < train_days * 0.8:
            current_test_end -= step
            continue
            
        windows.append({
            'train': df.iloc[train_start:val_start],
            'val': df.iloc[val_start:test_start],
            'test': df.iloc[test_start:current_test_end],
            'id': current_test_end,
            'dates': {
                'train': (df['dt'].iloc[train_start], df['dt'].iloc[val_start-1]),
                'val': (df['dt'].iloc[val_start], df['dt'].iloc[test_start-1]),
                'test': (df['dt'].iloc[test_start], df['dt'].iloc[current_test_end-1])
            }
        })
        current_test_end -= step
        if current_test_end < min_required:
            break
    
    return windows[::-1]

In [7]:
def select_top_features(model, feature_names, top_n):
    """Отбор топ фичей с обработкой различных типов моделей"""
    try:
        if hasattr(model, 'coef_') and model.coef_ is not None and len(model.coef_) > 0:
            coefs = np.abs(model.coef_)
            top_indices = np.argsort(coefs)[-top_n:]
        elif hasattr(model, 'feature_importances_') and model.feature_importances_ is not None:
            importances = model.feature_importances_
            top_indices = np.argsort(importances)[-top_n:]
        else:
            return feature_names[:min(top_n, len(feature_names))]
        
        return [feature_names[i] for i in top_indices if i < len(feature_names)]
    except:
        return feature_names[:min(top_n, len(feature_names))]

In [8]:
def make_objective(model_name, X_train, y_train, X_val, y_val, model_config):
    """Подбор гиперпараметров с улучшенной обработкой ошибок"""
    def objective(trial):
        try:
            params = {}
            if 'optuna_objective' in model_config and model_config['optuna_objective']:
                params = model_config['optuna_objective'](trial)
            
            if model_name == 'CatBoost':
                model = CatBoostRegressor(**params, verbose=False, random_state=42)
            elif model_name == 'MLP':
                # Специальная обработка для MLP
                h1 = params.pop('h1', 100)
                h2 = params.pop('h2', 50)
                params['hidden_layer_sizes'] = (h1, h2)
                model = MLPRegressor(**params, random_state=42, early_stopping=True, verbose=False)
            elif model_name == 'Stacking':
                # Stacking создается отдельно
                return float('inf')
            else:
                model = model_config['model']
                if params:
                    model.set_params(**params)
            
            model.fit(X_train, y_train)
            y_pred = model.predict(X_val)
            
            if metric_optuna == 'MAPE':
                score = mean_absolute_percentage_error(y_val, y_pred)
            elif metric_optuna == 'MAE':
                score = mean_absolute_error(y_val, y_pred)
            elif metric_optuna == 'RMSE':
                score = np.sqrt(mean_squared_error(y_val, y_pred))
            else:
                score = mean_absolute_percentage_error(y_val, y_pred)
            
            return score
        except Exception as e:
            return float('inf')
    
    return objective

In [9]:
def create_stacking_ensemble(X_train, y_train, X_val, y_val):
    """Создание stacking ансамбля из лучших моделей"""
    try:
        # Базовые модели для ансамбля
        base_models = [
            ('rf', RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)),
            ('xgb', XGBRegressor(n_estimators=100, max_depth=5, random_state=42, n_jobs=-1)),
            ('lgbm', LGBMRegressor(n_estimators=100, max_depth=5, random_state=42, n_jobs=-1, verbose=-1)),
            ('et', ExtraTreesRegressor(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1))
        ]
        
        # Финальная модель
        final_estimator = Ridge(alpha=1.0)
        
        stacking_model = StackingRegressor(
            estimators=base_models,
            final_estimator=final_estimator,
            cv=3,
            n_jobs=-1
        )
        
        stacking_model.fit(X_train, y_train)
        
        return stacking_model
    except Exception as e:
        print(f"Ошибка создания stacking ансамбля: {e}")
        return None

In [10]:
def train_val_test_ml_models(df, windows, models_config, ticker):
    """Обновленный цикл обучения с правильной фильтрацией колонок"""
    print(f'\n{"="*100}')
    print(f'Запущен цикл разработки ML моделей для прогнозирования стоимости акций "{ticker}"')
    print(f'{"="*100}\n')
    
    df_ml_db = pd.DataFrame(columns=[
        'dt', 'ticker', 'left_date', 'right_date', 'model_name', 
        'model_value', 'train_period', 'val_period', 'test_period', 
        'step', 'mape_mean', 'mae_mean', 'rmse_mean', 'r2_mean', 'best_features'
    ])
    
    df_ml = pd.DataFrame(columns=['test_period', 'model_name', 'mape', 'mae', 'rmse', 'r2'])
    
    # Фильтрация моделей которые можно обучить
    available_models = [m for m in models_config.keys() if m not in ['Stacking']]
    
    for model_name in available_models:
        ml_info = []
        model_config = models_config[model_name]
        
        print(f'\n{"-"*80}')
        print(f'Модель: {model_name}')
        print(f'{"-"*80}')
        
        for idx, window in enumerate(windows):
            train, val, test = window['train'], window['val'], window['test']
            
            # Проверка на достаточность данных
            if len(train) < 10 or len(test) < 3:
                print(f'⚠️ Окно {idx+1}: недостаточно данных, пропускаем')
                continue
            
            # 🔥 ВАЖНО: Фильтрация ТОЛЬКО числовых колонок
            feature_cols = [
                col for col in train.columns 
                if col not in ['dt', 'ticker', 'target', 'ticker_x', 'ticker_y'] 
                and train[col].dtype in ['int64', 'float64', 'int32', 'float32']
            ]
            
            if len(feature_cols) == 0:
                print(f'⚠️ Окно {idx+1}: нет числовых фичей для обучения')
                continue
            
            print(f'\nОкно {idx+1}/{len(windows)} | Фичей: {len(feature_cols)}')
            print(f"Обучение: {len(train)} дн. | Валидация: {len(val)} дн. | Тестирование: {len(test)} дн.")
            
            # Разделение на X и y
            X_train, y_train = train[feature_cols], train['target']
            X_val, y_val = val[feature_cols], val['target']
            X_test, y_test = test[feature_cols], test['target']
            
            # 🔥 Принудительная конвертация в float и заполнение пропусков
            X_train = X_train.astype(np.float64).fillna(0)
            X_val = X_val.astype(np.float64).fillna(0)
            X_test = X_test.astype(np.float64).fillna(0)
            
            # Стандартизация для определенных моделей
            scaler = None
            if model_name in ['SVR', 'MLP', 'KNeighbors', 'LinearRegression', 'Ridge']:
                scaler = StandardScaler()
                X_train_scaled = scaler.fit_transform(X_train)
                X_val_scaled = scaler.transform(X_val)
                X_test_scaled = scaler.transform(X_test)
            else:
                X_train_scaled = X_train.values
                X_val_scaled = X_val.values
                X_test_scaled = X_test.values
            
            # Оптимизация гиперпараметров
            best_params = {}
            if model_config.get('optuna_objective'):
                try:
                    objective = make_objective(model_name, X_train_scaled, y_train.values, 
                                             X_val_scaled, y_val.values, model_config)
                    study = optuna.create_study(direction='minimize')
                    study.optimize(objective, n_trials=min(n_trials, 10), show_progress_bar=False)
                    best_params = study.best_params
                    print(f"✓ Лучшие ГП: MAPE={study.best_value:.4f}")
                except Exception as e:
                    print(f"⚠️ Ошибка оптимизации: {e}")
                    best_params = {}
            
            # Обучение модели
            try:
                if model_name == 'CatBoost':
                    best_model = CatBoostRegressor(**best_params, verbose=False, random_state=42)
                elif model_name == 'MLP':
                    h1 = best_params.pop('h1', 100) if 'h1' in best_params else 100
                    h2 = best_params.pop('h2', 50) if 'h2' in best_params else 50
                    best_params['hidden_layer_sizes'] = (h1, h2)
                    best_model = MLPRegressor(**best_params, random_state=42, 
                                            early_stopping=True, verbose=False, max_iter=500)
                else:
                    best_model = model_config['model']
                    if best_params:
                        best_model.set_params(**best_params)
                
                # Обучение
                if scaler:
                    best_model.fit(X_train_scaled, y_train.values)
                else:
                    best_model.fit(X_train, y_train.values)
                
                # Отбор фичей
                selected_features = select_top_features(best_model, feature_cols, top_n_features)
                
                # Переобучение на топ фичах
                if len(selected_features) > 0:
                    if scaler:
                        selected_idx = [feature_cols.index(f) for f in selected_features if f in feature_cols]
                        best_model.fit(X_train_scaled[:, selected_idx], y_train.values)
                        y_test_pred = best_model.predict(X_test_scaled[:, selected_idx])
                    else:
                        best_model.fit(X_train[selected_features], y_train.values)
                        y_test_pred = best_model.predict(X_test[selected_features])
                else:
                    if scaler:
                        y_test_pred = best_model.predict(X_test_scaled)
                    else:
                        y_test_pred = best_model.predict(X_test)
                
                # Метрики
                test_mape = mean_absolute_percentage_error(y_test, y_test_pred)
                test_mae = mean_absolute_error(y_test, y_test_pred)
                test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
                test_r2 = r2_score(y_test, y_test_pred)
                
                print(f'Метрики: MAPE={test_mape:.4f}, MAE={test_mae:.2f}, RMSE={test_rmse:.2f}, R²={test_r2:.4f}')
                
                ml_info.append({
                    'model': best_model,
                    'features': selected_features,
                    'mape': test_mape,
                    'mae': test_mae,
                    'rmse': test_rmse,
                    'r2': test_r2,
                    'scaler': scaler
                })
                
            except Exception as e:
                print(f"⚠️ Ошибка обучения модели: {e}")
                import traceback
                traceback.print_exc()
                continue
        
        if len(ml_info) == 0:
            print(f"⚠️ Модель {model_name} не удалось обучить ни на одном окне")
            continue
        
        # Выбор лучшей модели
        best_model_info = sorted(ml_info, key=lambda x: x['mape'])[0]
        best_model = best_model_info['model']
        
        # Объединение фичей
        all_features = set()
        for info in ml_info:
            all_features.update(info['features'])
        best_features = list(all_features)[:top_n_features]
        
        print(f'\n✓ Лучшая {model_name}: {len(best_features)} фичей')
        
        # Финальная оценка
        mape_list, mae_list, rmse_list, r2_list = [], [], [], []
        
        for idx, window in enumerate(windows):
            train, test = window['train'], window['test']
            
            if len(best_features) == 0:
                continue
            
            available_features = [f for f in best_features if f in train.columns and f in test.columns]
            if len(available_features) == 0:
                continue
            
            X_train_final = train[available_features].fillna(0).astype(np.float64)
            X_test_final = test[available_features].fillna(0).astype(np.float64)
            y_train_final = train['target'].values
            y_test_final = test['target'].values
            
            try:
                if best_model_info['scaler']:
                    X_train_final = best_model_info['scaler'].transform(X_train_final)
                    X_test_final = best_model_info['scaler'].transform(X_test_final)
                
                best_model.fit(X_train_final, y_train_final)
                y_pred = best_model.predict(X_test_final)
                
                mape = mean_absolute_percentage_error(y_test_final, y_pred)
                mae = mean_absolute_error(y_test_final, y_pred)
                rmse = np.sqrt(mean_squared_error(y_test_final, y_pred))
                r2 = r2_score(y_test_final, y_pred)
                
                mape_list.append(mape)
                mae_list.append(mae)
                rmse_list.append(rmse)
                r2_list.append(r2)
                
                new_row = pd.Series([
                    f"{window['dates']['test'][0]} - {window['dates']['test'][1]}",
                    model_name, round(mape, 4), round(mae, 4), round(rmse, 4), round(r2, 4)
                ], index=df_ml.columns)
                df_ml = pd.concat([df_ml, new_row.to_frame().T], ignore_index=True)
                
            except Exception as e:
                print(f"⚠️ Ошибка оценки окна {idx+1}: {e}")
                continue
        
        # Сохранение результатов
        if len(mape_list) > 0:
            new_row_db = pd.Series([
                datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
                ticker, left_date, right_date, model_name,
                str(type(best_model).__name__),
                train_period, val_period, test_period, step,
                float(round(np.mean(mape_list), 4)),
                float(round(np.mean(mae_list), 4)),
                float(round(np.mean(rmse_list), 4)),
                float(round(np.mean(r2_list), 4)),
                json.dumps(best_features)
            ], index=df_ml_db.columns)
            df_ml_db = pd.concat([df_ml_db, new_row_db.to_frame().T], ignore_index=True)
        
        print(f'Средние метрики: MAPE={np.mean(mape_list):.4f}, MAE={np.mean(mae_list):.2f}')
    
    # Отправка в БД
    try:
        df_ml_db.to_sql('final_ml_models_data', con=connection(), if_exists='append', index=False)
        print(f'\n✓ Результаты сохранены в БД: {len(df_ml_db)} записей')
    except Exception as e:
        print(f'⚠️ Ошибка сохранения в БД: {e}')
    
    return df_ml, df_ml_db

In [22]:
final_ml_models_data_tickers = list(pd.read_sql("SELECT DISTINCT ticker FROM final_ml_models_data", connection())['ticker'])

In [24]:
results_summary = []

for ticker in tickers:
    try:
        if ticker in final_ml_models_data_tickers:
            continue

        print(f"\n{'#' * 100}")
        print(f"# ТИКЕР: {ticker}")
        print(f"{'#' * 100}\n")

        # Сбор данных
        data = pack_all_data_for_ml_models(ticker, left_date, right_date, connection())

        if len(data) < 50:
            print(f"⚠️ Недостаточно данных для {ticker} ({len(data)} записей)")
            continue

        # Сохранение сырых данных
        data.to_csv(f"{os.getcwd()}/all_features_{ticker}_2026.csv", index=False)

        # Подготовка
        df = data.copy()
        df = df[~df["target"].isnull()]

        # Проверка на числовые колонки
        numeric_cols = df.select_dtypes(include=[np.number]).columns
        print(f"✓ Найдено {len(numeric_cols)} числовых колонок")

        # Создание окон
        windows = sliding_windows_cross_validatin(
            df, train_period, val_period, test_period, step
        )

        if len(windows) == 0:
            print(f"⚠️ Не удалось создать окна кросс-валидации для {ticker}")
            continue

        print(f"✓ Создано {len(windows)} окон кросс-валидации")

        # Обучение моделей
        df_ml, df_ml_db = train_val_test_ml_models(df, windows, MODELS_CONFIG, ticker)

        # Сохранение результатов
        if len(df_ml) > 0:
            best_model = df_ml.loc[df_ml["mape"].idxmin()]
            results_summary.append(
                {
                    "ticker": ticker,
                    "best_model": best_model["model_name"],
                    "best_mape": best_model["mape"],
                    "best_mae": best_model["mae"],
                    "windows": len(windows),
                }
            )

        print(f"\n✓ Обработка {ticker} завершена\n")

    except Exception as e:
        print(f"\n⚠️ Ошибка при работе с тикером {ticker}: {e}")
        import traceback

        traceback.print_exc()
        continue

# Итоговая сводка
if len(results_summary) > 0:
    summary_df = pd.DataFrame(results_summary)
    print(f"\n{'=' * 100}")
    print("ИТОГОВАЯ СВОДКА ПО ВСЕМ ТИКЕРАМ")
    print(f"{'=' * 100}")
    print(summary_df.sort_values("best_mape").to_string(index=False))
    print(f"\nВсего обработано тикеров: {len(results_summary)}")
    print(f"Средний MAPE лучших моделей: {summary_df['best_mape'].mean():.4f}")


####################################################################################################
# ТИКЕР: CBOM
####################################################################################################

⚠️ Недостаточно данных для CBOM (43 записей)

####################################################################################################
# ТИКЕР: BSPB
####################################################################################################

⚠️ Недостаточно данных для BSPB (45 записей)

####################################################################################################
# ТИКЕР: TATN
####################################################################################################

⚠️ Недостаточно данных для TATN (48 записей)

####################################################################################################
# ТИКЕР: TATNP
##############################################################################################

Traceback (most recent call last):
  File "/var/folders/3y/q4zxlyrs3md__fhhs8mw34qc0000gn/T/ipykernel_45901/2625846943.py", line 92, in train_val_test_ml_models
    best_model = MLPRegressor(**best_params, random_state=42,
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
TypeError: sklearn.neural_network._multilayer_perceptron.MLPRegressor() got multiple values for keyword argument 'max_iter'


✓ Лучшие ГП: MAPE=0.7459
⚠️ Ошибка обучения модели: sklearn.neural_network._multilayer_perceptron.MLPRegressor() got multiple values for keyword argument 'max_iter'
⚠️ Модель MLP не удалось обучить ни на одном окне

✓ Результаты сохранены в БД: 8 записей

✓ Обработка MOEX завершена


####################################################################################################
# ТИКЕР: AFKS
####################################################################################################



Traceback (most recent call last):
  File "/var/folders/3y/q4zxlyrs3md__fhhs8mw34qc0000gn/T/ipykernel_45901/2625846943.py", line 92, in train_val_test_ml_models
    best_model = MLPRegressor(**best_params, random_state=42,
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
TypeError: sklearn.neural_network._multilayer_perceptron.MLPRegressor() got multiple values for keyword argument 'max_iter'


⚠️ Недостаточно данных для AFKS (48 записей)

####################################################################################################
# ТИКЕР: LSRG
####################################################################################################

⚠️ Недостаточно данных для LSRG (42 записей)

####################################################################################################
# ТИКЕР: RASP
####################################################################################################

⚠️ Недостаточно данных для RASP (35 записей)

####################################################################################################
# ТИКЕР: SVAV
####################################################################################################

⚠️ Недостаточно данных для SVAV (33 записей)

####################################################################################################
# ТИКЕР: ENPG
##################################################

...

✓ Найдено 7647 числовых колонок
✓ Создано 2 окон кросс-валидации

Запущен цикл разработки ML моделей для прогнозирования стоимости акций "POSI"


--------------------------------------------------------------------------------
Модель: DecisionTree
--------------------------------------------------------------------------------

Окно 1/2 | Фичей: 7646
Обучение: 21 дн. | Валидация: 14 дн. | Тестирование: 7 дн.
✓ Лучшие ГП: MAPE=0.0350
Метрики: MAPE=0.0943, MAE=116.30, RMSE=117.61, R²=-44.1426

Окно 2/2 | Фичей: 7646
Обучение: 21 дн. | Валидация: 14 дн. | Тестирование: 7 дн.
✓ Лучшие ГП: MAPE=0.0570
Метрики: MAPE=0.0779, MAE=93.35, RMSE=94.37, R²=-45.5283

✓ Лучшая DecisionTree: 500 фичей
Средние метрики: MAPE=0.0861, MAE=104.83

--------------------------------------------------------------------------------
Модель: RandomForest
--------------------------------------------------------------------------------

Окно 1/2 | Фичей: 7646
Обучение: 21 дн. | Валидация: 14 дн. | Тестирование: 7 д

Traceback (most recent call last):
  File "/var/folders/3y/q4zxlyrs3md__fhhs8mw34qc0000gn/T/ipykernel_45901/2625846943.py", line 92, in train_val_test_ml_models
    best_model = MLPRegressor(**best_params, random_state=42,
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
TypeError: sklearn.neural_network._multilayer_perceptron.MLPRegressor() got multiple values for keyword argument 'max_iter'


✓ Лучшие ГП: MAPE=0.9872
⚠️ Ошибка обучения модели: sklearn.neural_network._multilayer_perceptron.MLPRegressor() got multiple values for keyword argument 'max_iter'
⚠️ Модель MLP не удалось обучить ни на одном окне

✓ Результаты сохранены в БД: 8 записей

✓ Обработка POSI завершена


####################################################################################################
# ТИКЕР: MDMG
####################################################################################################



Traceback (most recent call last):
  File "/var/folders/3y/q4zxlyrs3md__fhhs8mw34qc0000gn/T/ipykernel_45901/2625846943.py", line 92, in train_val_test_ml_models
    best_model = MLPRegressor(**best_params, random_state=42,
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
TypeError: sklearn.neural_network._multilayer_perceptron.MLPRegressor() got multiple values for keyword argument 'max_iter'


⚠️ Недостаточно данных для MDMG (48 записей)

####################################################################################################
# ТИКЕР: ASTR
####################################################################################################

⚠️ Недостаточно данных для ASTR (47 записей)

####################################################################################################
# ТИКЕР: UWGN
####################################################################################################

⚠️ Недостаточно данных для UWGN (45 записей)

####################################################################################################
# ТИКЕР: TRMK
####################################################################################################

⚠️ Недостаточно данных для TRMK (39 записей)

####################################################################################################
# ТИКЕР: RENI
##################################################